# Chapter 12. Pitch-trajectory preparation

**Learning objective.** Preserve the denominator of requested frames, assign distinct missingness causes, retain three time coordinates, and distinguish observed from bounded interpolation.

**Estimated time:** 75–90 minutes. **Exercises:** 12.1 and 12.2. All 96 trajectories and their F0 values are explicitly **synthetic teaching fixtures**; they are not observations of speakers.

## Input and output contract

Inputs are source and interval manifests, a fixed-delay landmark, raw frame candidates, and the trajectory specification. The notebook reconstructs every requested 10 ms frame, verifies the join, permits interpolation only for estimator-failure gaps no longer than 20 ms, and writes requested frames, preprocessing state, coverage with contributing-token counts, diagnostics, decisions, and provenance.

In [ ]:
find_repository <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "companion", "data", "MANIFEST.tsv"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Run from the repository root or a subdirectory.")
    current <- parent
  }
}
REPOSITORY_ROOT <- find_repository()
DATA_ROOT <- file.path(REPOSITORY_ROOT, "companion", "data", "ch12", "synthetic_prosody")
if (!dir.exists(DATA_ROOT)) stop("Chapter 12 fixture absent. Run companion/scripts/generate_ch12_fixture.py explicitly; no data are substituted silently.")
source(file.path(REPOSITORY_ROOT, "companion", "r", "ch12.R"))
cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated contours and intervals, not human prosody.
")

In [ ]:
OUTPUT <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch12", "01_pitch_trajectory_preparation")
result <- run_pitch_trajectory_preparation(file.path(DATA_ROOT, "source_manifest.tsv"), file.path(DATA_ROOT, "interval_manifest.tsv"), file.path(DATA_ROOT, "landmarks.tsv"), file.path(DATA_ROOT, "raw_frame_estimates.tsv"), file.path(DATA_ROOT, "prosodic_trajectory_spec.yaml"), OUTPUT)
stopifnot(nrow(result$prepared) > 3000, result$raw_unchanged)
stopifnot(all(c("structural_unvoiced", "estimator_failure", "quality_rejection", "source_failure") %in% result$prepared$gap_code))
stopifnot(any(result$prepared$value_origin == "interpolated_estimator_failure"))
print(head(result$coverage, 20))
cat("Requested frames:", nrow(result$prepared), "
")

## Learner decision

Compare denominators obtained from requested frames and successful estimates. Plot the same synthetic contours in onset-relative milliseconds, proportional time, and time from the fixed-delay event. State which claim each coordinate can support and which it obscures. Keep structural voicelessness separate from tracker failure and interpolation.

In [ ]:
stopifnot(all(file.exists(file.path(OUTPUT, c("requested_frames.tsv", "trajectory_preprocessing.tsv", "trajectory_coverage.tsv", "trajectory_diagnostics.html", "preprocessing_decision.yaml", "run_provenance.json")))))
print(result$decision)